In [17]:
import optuna
import pandas as pd
import plotly.express as px
pd.set_option('display.float_format', lambda x: '%.2f' % x)


from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, pairwise_distances
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from  scipy.stats import bartlett, shapiro
from pingouin import welch_anova



### Carregar Dados

In [4]:
df_clientes = pd.read_csv('./dados/dataset_clientes_pj.csv')

In [5]:
df_clientes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 6 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   atividade_economica     500 non-null    object 
 1   faturamento_mensal      500 non-null    float64
 2   numero_de_funcionarios  500 non-null    int64  
 3   localizacao             500 non-null    object 
 4   idade                   500 non-null    int64  
 5   inovacao                500 non-null    int64  
dtypes: float64(1), int64(3), object(2)
memory usage: 23.6+ KB


In [6]:
df_clientes.head(10)

,atividade_economica,faturamento_mensal,numero_de_funcionarios,localizacao,idade,inovacao
0,Comércio,713109.95,12,Rio de Janeiro,6,1
1,Comércio,790714.38,9,São Paulo,15,0
2,Comércio,1197239.33,17,São Paulo,4,9
3,Indústria,449185.78,15,São Paulo,6,0
4,Agronegócio,1006373.16,15,São Paulo,15,8
5,Serviços,1629562.41,16,Rio de Janeiro,11,4
6,Serviços,771179.95,13,Vitória,0,1
7,Serviços,707837.61,16,São Paulo,10,6
8,Comércio,888983.66,17,Belo Horizonte,10,1
9,Indústria,1098512.64,13,Rio de Janeiro,9,3


### EDA

In [7]:
                                                                                                                                                                                                                            # Distriuição da variavel Inovação
percentual_inovacao = df_clientes.value_counts('inovacao') / len(df_clientes) * 100
px.bar(percentual_inovacao, color=percentual_inovacao.index)

In [8]:
    """
  Teste ANOVA (análise de variancia)
  Verificar se há variações significativas na média de faturamento mensal para diferentes níveis de inovação.
  Suposições / Pressupostos:
  - Obsorvações independentes
  - Variável dependente é continua
  - Segue uma distribuição normal
  - Homogeneidade de variâncias
  - Amostras sejam de tamanhos iguais  
    """


'\n  Teste ANOVA (análise de variancia)\n  Verificar se há variações significativas na média de faturamento mensal para diferentes níveis de inovação.\n  Suposições / Pressupostos:\n  - Obsorvações independentes\n  - Variável dependente é continua\n  - Segue uma distribuição normal\n  - Homogeneidade de variâncias\n  - Amostras sejam de tamanhos iguais  \n'

In [13]:
""" Checar se as variancias (faturamento) entre os grupos (inovação) são homogêneas
    Aplicar Teste de Barlett
    H0 variancias são iguais
    H1 variancias são diferentes
    
"""
# Separando os dados de faturamento em grupos com base ma coluna de inovação
dados_agrupados = [df_clientes['faturamento_mensal'][df_clientes['inovacao'] == grupo] for grupo in df_clientes['inovacao'].unique()]

# Executar o teste de Bartlett
bartlett_test_statistic, bartlett_p_value = bartlett(*dados_agrupados)

# Exibir os resultados
print(f'Estatistica do Teste de Bartlett: {bartlett_test_statistic:.4f}')
print(f'P-Value do Teste de Bartlett: {bartlett_p_value:.4f}')

Estatistica do Teste de Bartlett: 10.9012
P-Value do Teste de Bartlett: 0.2825


In [16]:
 """
 Executar o Teste de Shapiro-Wilk para verificar a normalidade dos dados
 Verificar se os dados seguem uma distribuição normal
 H0 Segue uma distribuição normal
 H1 Não segue uma distribuição normal

 """

# Executar o teste de Shapiro
shapiro_test_statistic, shapiro_p_value = shapiro(df_clientes['faturamento_mensal'])

# Exibir os resultados
print(f'Estatistica do Teste de shapiro: {shapiro_test_statistic:.4f}')
print(f'P-Value do Teste de shapiro: {shapiro_p_value:.4f}')


Estatistica do Teste de shapiro: 0.9960
P-Value do Teste de shapiro: 0.2351


In [18]:
""" 
Aplicar a ANOVA de Welch, pois as amostras são de tamanhos diferentes
H0 não há diferença significativa entre as médias dos grupos
H1 há pelo menos uma diferença significativa entre as médias dos grupos

"""
aov = welch_anova(dv='faturamento_mensal', between='inovacao', data=df_clientes)

# Exibir os resultados
print(f'Estatistica do Teste de ANOVA de Welch: {aov.loc[0, 'F']:.4f}')
print(f'P-Value do Teste de ANOVA de Welch: {aov.loc[0, "p-unc"]:.4f}')

Estatistica do Teste de ANOVA de Welch: 1.1270
P-Value do Teste de ANOVA de Welch: 0.3453
